# Phase 2.5 Scenario Memory Evaluation

Thin notebook for running the scenario-sliced temporal-memory evaluator and inspecting artifacts. Metrics are future-pseudo projected-depth occupancy metrics, not dense simulator 3D ground-truth IoU.

Scenario slices are heuristic and non-mutually exclusive. The `vehicle_change` slice is selected from scene semantic occupancy masks, while IoU rows are limited to the configured `class_ids`.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from src.evaluation.phase2_carla_eval import (
    ScenarioMemoryEvalConfig,
    run_scenario_memory_evaluation,
)

OUTPUT_DIR = Path("outputs/phase2/scenario_memory")
MAX_SCAN_FRAMES = 2000
ANCHORS_PER_SLICE = 50
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Run Scenario Memory Evaluation


In [ ]:
config = ScenarioMemoryEvalConfig(
    max_scan_frames=MAX_SCAN_FRAMES,
    anchors_per_slice=ANCHORS_PER_SLICE,
    output_dir=OUTPUT_DIR,
    bev_frame_count_per_slice=2,
)

artifacts = run_scenario_memory_evaluation(config=config)
for name, path in artifacts.__dict__.items():
    print(f"{name}: {path}")


## 3. Selected Anchor Counts


In [ ]:
summary = json.loads(artifacts.eval_summary.read_text(encoding="utf-8"))
counts = pd.DataFrame(
    [
        {"slice": slice_name, "selected_anchor_count": count}
        for slice_name, count in summary["selected_anchor_counts"].items()
    ]
)
display(counts)
print(json.dumps(summary["selection_thresholds"], indent=2))


## 4. Per-Slice IoU


In [ ]:
by_slice = json.loads(artifacts.by_slice_iou.read_text(encoding="utf-8"))
rows = []
for slice_name, payload in by_slice["slices"].items():
    for row in payload["classes"]:
        rows.append(
            {
                "slice": slice_name,
                "class_name": row["class_name"],
                "baseline_iou": row["baseline_iou"],
                "temporal_iou": row["temporal_iou"],
                "delta": row["delta"],
                "baseline_union": row["baseline_union"],
                "temporal_union": row["temporal_union"],
                "informative": row["informative"],
            }
        )

df = pd.DataFrame(rows)
display(
    df.style.format(
        {
            "baseline_iou": "{:.3f}",
            "temporal_iou": "{:.3f}",
            "delta": "{:+.3f}",
        }
    )
)


## 5. Selected Anchors


In [ ]:
selected_records = [
    json.loads(line)
    for line in artifacts.selected_anchors.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
selected_df = pd.DataFrame(selected_records)
display(selected_df.head(20))


## 6. BEV Visual Checks


In [ ]:
for image_path in artifacts.bev_images:
    display(Markdown(f"**{Path(image_path).parent.name} / {Path(image_path).name}**"))
    display(Image(filename=str(image_path)))


## 7. Interpretation Notes


In [ ]:
notes = []
for slice_name, slice_df in df.groupby("slice"):
    vehicle_rows = slice_df[slice_df["class_name"] == "vehicle"]
    if vehicle_rows.empty:
        continue
    vehicle = vehicle_rows.iloc[0]
    if not bool(vehicle["informative"]):
        notes.append(f"{slice_name}: vehicle IoU is non-informative because union is empty.")
    elif vehicle["delta"] > 0:
        notes.append(f"{slice_name}: temporal fusion improves vehicle future-pseudo IoU by {vehicle['delta']:+.3f}.")
    elif vehicle["delta"] == 0:
        notes.append(f"{slice_name}: temporal fusion preserves vehicle future-pseudo IoU.")
    else:
        notes.append(f"{slice_name}: temporal fusion reduces vehicle future-pseudo IoU by {vehicle['delta']:+.3f}.")

notes.append("Inspect BEV examples before making any qualitative claim; this notebook is diagnostic, not a final benchmark.")
display(Markdown("\n".join(f"- {note}" for note in notes)))
